## Libraries & Function

In [2]:
import numpy as np
from scipy.linalg import expm
from qutip import *
import numba
from numba import njit, prange
import os
import time

In [3]:

sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) ; sm = np.array(([[0,1],[0,0]]), dtype=complex) ; sp = np.array(([[0,0],[1,0]]), dtype=complex)


In [4]:
#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

### Hamiltonians and U operator

In [ ]:
def interaction_Hamiltonian(c_CM, P12, P21, sp, sm):
    """
    Builds the Interaction Hamiltonian for the 3-level System and Ancilla collision.
    Implements the model: c * (|1><2| @ sigma_plus_a + |2><1| @ sigma_minus_a)
        
    Parameters: 
    - c_CM : float, Interaction strength for the collision
    - P12_sys : numpy array, System relaxes from |2> to |1>
    - P21_sys : numpy array, System excites from |1> to |2>
    - sp : numpy array, Ancilla raising operator
    - sm : numpy array, Ancilla lowering operator
        
    Returns: 
    - H_int : numpy array (6x6), Interaction Hamiltonian
    """
    # Tensor product for relaxation: System relaxes (P12), Ancilla excites (sp)
    term_relaxation = np.kron(P12_sys, sp)
    
    # Tensor product for excitation: System excites (P21), Ancilla relaxes (sm)
    term_excitation = np.kron(P21_sys, sm)
    
    # Total Collisional Hamiltonian
    H_int = c_CM * (term_relaxation + term_excitation)
    
    return H_int

In [ ]:
def complete_Hamiltonian(H_Sys, c_CM, P12, P21, sp, sm):
    """
    Generates the Hamiltonians for the 3-level system collision model using pure NumPy:
                - H_system : system Hamiltonian
                - H_collision : interaction Hamiltonian with 1 ancilla
                - H_tot : complete Hamiltonian (system + collision)

    Parameters: 
    - H_Sys: numpy array (3x3), System Hamiltonian
    - c_CM : float, Interaction Force
    - P12_sys, P21_sys, sp, sm : numpy arrays, operators for the interaction
                
    Returns: 
    - H_system, H_collision, H_tot (all as numpy arrays)
    """
    # 1. System Hamiltonian 
    H_system = H_Sys
    
    # 2. Collision Hamiltonian
    H_collision = interaction_Hamiltonian(c_CM, P12, P21, sp, sm) 
    
    # 3. Total Hamiltonian
    # Expand H_sys in the total space: System (3x3) tensor Identity_ancilla (2x2)
    Id_ancilla = np.eye(2, dtype=complex)
    H_system_expanded = np.kron(H_system, Id_ancilla)  
        
    H_tot = H_system_expanded + H_collision
        
    return H_system, H_collision, H_tot

In [ ]:
def evolution_operator(H, dt, method='expm', hermitian=True):
    """
    Build up of the evolution operator U = exp(-i H dt) using Expm or analytic diagonalization.
   
    Parameters: - H : Qobj or nparray, System Hamiltonian
                - dt : float, Timestep
    
    Method : - "expm"-> build up of the Matrix Exponential with expm
             - "diagonalization"->  build up of the propagater U as V @(exp(-i W dt))@ V_dag with W eigenvalues and V eigenvector of the Hamiltonian 

    Returns : Evolution Operator U, 
    """
    H = H.full() if hasattr(H, "full") else np.array(H)
    
    # -----------
    # Expm method
    # -----------
    
    if method == 'expm':
        U = expm(-1j * H * dt)
        return U
        
    # ---------------
    # Diagonalization
    # ---------------
    
    elif method == 'diagonalization':
        if hermitian:
            w, V = np.linalg.eigh(H)
            V_inv = V.conj().T
        else:
            w, V = np.linalg.eig(H) 
            V_inv = np.linalg.inv(V)
                
        U_diag = np.diag(np.exp(-1j * w * dt))
        U = V @ U_diag @ V_inv
        return U, U_diag, w, V

    else:
        raise ValueError("method must be 'expm' or 'diagonalization'")


### Lindblad functions

In [ ]:
def Liouvillian(H, gamma_k, L_k):
    """
    Build the Liouvillian superoperator using row-major convention (NumPy).
    
    Parameters: - H : nparray, Hamiltonian matrix
                - gamma_k : list, Decay rates
                - L_k : list, Jump Operators
    
    Returns: - super_L : nparray, Liouvillian superoperator
    """    
    I = np.eye(H.shape[0], dtype=complex)
    
    # Unitary evolution: -i * [H, rho]
    super_L = -1.j * (np.kron(H, I) - np.kron(I, H.T))
    
    # Dissipator terms
    for k in range(len(gamma_k)):
        L = L_k[k]
        L_dag = np.conj(L).T
        L_dag_L = L_dag @ L
        
        super_L += gamma_k[k] * (np.kron(L, np.conj(L)) - 0.5 * np.kron(L_dag_L, I) - 0.5 * np.kron(I, L_dag_L.T))
    
    return super_L


In [ ]:
@njit(cache=True)
def _evolve_expm_core(super_U, rho_vec_initial, n_times):
    """
    Core evolution loop with expm method (Numba JIT)
    """
    rho_size = rho_vec_initial.shape[0]
    rho_vec_list = np.zeros((rho_size, n_times), dtype=np.complex128)
    rho_vec_list[:, 0] = rho_vec_initial
    
    for i in range(1, n_times):
        rho_vec_list[:, i] = super_U @ rho_vec_list[:, i - 1]
    
    return rho_vec_list


@njit(cache=True)
def _evolve_diagonal_core(V, V_inv, U_diag, rho_vec_initial, n_times):
    """
    Core evolution loop with diagonal method (Numba JIT)
    """
    n_states = len(U_diag)
    
    # Initial coefficients in eigenbasis
    coeff = V_inv @ rho_vec_initial
    coeff_list = np.zeros((n_states, n_times), dtype=np.complex128)
    coeff_list[:, 0] = coeff
    
    # Evolution of coefficients
    for i in range(1, n_times):
        coeff_list[:, i] = U_diag * coeff_list[:, i - 1]
    
    # Transform back to original basis
    rho_vec_list = V @ coeff_list
    
    return rho_vec_list


def Lindblad_evo(rho, H, gamma_k, L_k, times, method="expm", vectorized=True):
    """
    Evolution of the density matrix with the Lindblad Eq. (Optimized with Numba)
    
    Method: - "expm" -> propagator = expm(super_L * dt)
            - "diagonal" -> diagonalization of the super-operator
        
    Vectorized: True/False to choose the output format
    
    Parameters: - H : nparray, System Hamiltonian
                - rho : Qobj or nparray, Initial Density Matrix
                - gamma_k : list, List of Decay Rates
                - L_k : list, List of Jump Operators
                - times : array, Time array
        
    Returns : - if vectorized=True → array (N^2, Nt)
              - if vectorized=False → array (Nt, N_site, N_site)
              - if method="diagonal" also returns V, W
    """
    # Convert to NumPy
    L_k = [L.full() if hasattr(L, "full") else np.array(L, dtype=complex) for L in L_k]
    H = H.full() if hasattr(H, "full") else np.array(H, dtype=complex)
    rho = rho.full() if hasattr(rho, "full") else np.array(rho, dtype=complex)
    
    rho_shape = H.shape[0]
    dt = times[1] - times[0]
    n_times = len(times)
    
    # Build Liouvillian
    super_L = Liouvillian(H, gamma_k, L_k)
    
    # Vectorize initial state
    rho_vec = rho.reshape(rho_shape * rho_shape)
    
    # -------------
    # Expm method
    # -------------
    if method == "expm":
        # Compute propagator 
        super_U = expm(super_L * dt)
        
        # evolution loop
        rho_vec_list = _evolve_expm_core(super_U, rho_vec, n_times)
        
        # Output
        if vectorized:
            return rho_vec_list
        else:
            return rho_vec_list.T.reshape(n_times, rho_shape, rho_shape)
    
    # ------------------
    # Diagonal method
    # ------------------
    elif method == "diagonal":
        # Diagonalize Liouvillian 
        W, V = np.linalg.eig(super_L)
        V_inv = np.linalg.inv(V)
        
        # Diagonal evolution operator
        U_diag = np.exp(W * dt)
        
        # evolution loop
        rho_vec_list = _evolve_diagonal_core(V, V_inv, U_diag, rho_vec, n_times)
        
        # Output
        if vectorized:
            return rho_vec_list, V, W
        else:
            return rho_vec_list.T.reshape(n_times, rho_shape, rho_shape), V, W
    
    else:
        raise ValueError("method must be 'expm' or 'diagonal'")

### Isolated system

In [ ]:
@njit(cache=True)
def _compute_trajectory_isolated_core_single(psi_initial, U_site, projectors, projectors_cohe, n_times):
    """
    Core evolution loop optimized with Numba for a single qubit.
    Computes both populations (real) and coherences (complex) over time.
    """
    N_proj = projectors.shape[0]
    N_cohe = projectors_cohe.shape[0]
    
    # Pre-allocate arrays
    pop_traj = np.zeros((N_proj, n_times), dtype=np.float64)
    coh_traj = np.zeros((N_cohe, n_times), dtype=np.complex128)
    
    # Initial state values
    for p in range(N_proj):
        P_psi = projectors[p] @ psi_initial
        pop_traj[p, 0] = np.real(np.vdot(psi_initial, P_psi))
        
    for c in range(N_cohe):
        P_cohe_psi = projectors_cohe[c] @ psi_initial
        coh_traj[c, 0] = np.vdot(psi_initial, P_cohe_psi)
    
    # Time Evolution
    psi = psi_initial.copy()
    for step in range(1, n_times):
        psi = U_site @ psi
        
        # Store populations and coherences for the current step
        for p in range(N_proj):
            P_psi = projectors[p] @ psi
            pop_traj[p, step] = np.real(np.vdot(psi, P_psi))
            
        for c in range(N_cohe):
            P_cohe_psi = projectors_cohe[c] @ psi
            coh_traj[c, step] = np.vdot(psi, P_cohe_psi)

    return pop_traj, coh_traj

def compute_trajectory_wf_isolated(times, projectors, projectors_cohe, psi_sys_initial, U_site):
    """
    Optimized isolated system evolution with Numba for a single qubit.
    Calculates both populations and coherences.
    """
    # Convert to NumPy
    U_site_np = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_initial_np = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    # Flatten if needed
    if psi_initial_np.ndim > 1:
        psi_initial_np = psi_initial_np.flatten()

    # Time parameters
    n_times = len(times)
        
    # Call JIT-compiled function
    pop_traj_isolated, coh_traj_isolated = _compute_trajectory_isolated_core_single(
        psi_initial_np, U_site_np, projectors, projectors_cohe, n_times
    )

    return pop_traj_isolated, coh_traj_isolated

### Collisional Method functions

#### Evolution with $ U_{complete} $ and then trace on the ancilla

In [ ]:
@njit(cache=True)
def _compute_trace_ancilla_core_single(rho_sys, rho_anc, U_step, U_step_dag, projectors, projectors_cohe, n_times, dim_sys, dim_anc):
    """
    Core computation optimized with Numba for a single qubit + single ancilla.
    """
    N_state = projectors.shape[0]
    pops_complete = np.zeros((N_state, n_times), dtype=np.float64)
    cohe_complete = np.zeros((N_state, n_times), dtype=np.complex128)
    
    # Initial state expectations for all projectors
    for p in range(N_state):
        pops_complete[p, 0] = np.real(np.trace(projectors[p] @ rho_sys))
        cohe_complete[p, 0] = np.trace(projectors_cohe[p] @ rho_sys)
    
    # Time Evolution
    for t in range(1, n_times):
        # 1: Expansion (System tensor Ancilla)
        rho_tot = np.kron(rho_sys, rho_anc)
        
        # 2: Evolution
        rho_tot = U_step @ rho_tot @ U_step_dag
        
        # 3: Partial Trace over the Ancilla
        rho_tot_reshaped = rho_tot.reshape(dim_sys, dim_anc, dim_sys, dim_anc)
        
        # Manual trace 
        rho_sys = np.zeros((dim_sys, dim_sys), dtype=np.complex128)
        for i in range(dim_sys):
            for j in range(dim_sys):
                for k in range(dim_anc):
                    rho_sys[i, j] += rho_tot_reshaped[i, k, j, k]
        
        # 4: Store populations for the current step
        for p in range(N_state):
            pops_complete[p, t] = np.real(np.trace(projectors[p] @ rho_sys))
            cohe_complete[p, t] = np.trace(projectors_cohe[p] @ rho_sys)
    
    return pops_complete, cohe_complete


def compute_trace_ancilla(rho_sys_initial, rho_anc_single, U_diag, V, times, projectors, projectors_cohe):
    """
    Evolution with complete collisional Hamiltonian and then trace on the Ancilla 
    degrees of freedom for a single qubit system. Corresponds to an average 
    over an infinite number of trajectories.
    """

    # For a single qubit system, the total ancilla is just the single ancilla state
    rho_anc = rho_anc_single.full() if hasattr(rho_anc_single, 'full') else rho_anc_single.copy()
    
    # Convert system state to numpy
    rho_sys = rho_sys_initial.full() if hasattr(rho_sys_initial, 'full') else rho_sys_initial.copy()

    # Time parameters
    n_times = len(times)

    # Dimensions
    dim_sys = rho_sys.shape[0]
    dim_anc = rho_anc.shape[0]
    
    # Evolution operator for a single time step
    V_np = V.full() if hasattr(V, 'full') else V
    U_diag_np = U_diag.full() if hasattr(U_diag, 'full') else U_diag
    U_step = V_np @ U_diag_np @ V_np.conj().T
    U_step_dag = U_step.conj().T
    
    # Call JIT-compiled function
    pops_complete, cohe_complete = _compute_trace_ancilla_core_single(
        rho_sys, rho_anc, U_step, U_step_dag, projectors, projectors_cohe, n_times, dim_sys, dim_anc
    )
    
    return pops_complete, cohe_complete

#### Single trajectory Quantum Jump

In [ ]:
@njit
def sigma_xyz_expectation_value(psi, sx, sy, sz):
    """
    Function to calculate expectation value of <sigmax>, <sigmay>, <sigmaz> 
    using matrix multiplication for a single qubit.

    Parameters: - psi : nparray, wave function at time t (shape: 2,)
                - sx : nparray, Pauli X operator matrix
                - sy : nparray, Pauli Y operator matrix
                - sz : nparray, Pauli Z operator matrix

    returns : - S_x, float expectation value of <sigmax>
              - S_y, float expectation value of <sigmay>
              - S_z, float expectation value of <sigmaz>
    """
    
    S_x = np.real(np.vdot(psi, sx @ psi))
    S_y = np.real(np.vdot(psi, sy @ psi))
    S_z = np.real(np.vdot(psi, sz @ psi))

    return S_x, S_y, S_z

In [ ]:
@njit
def compute_Bloch_Sphere(psi):
    """
    Function to extract the expectation value of the Bloch's Sphere components 
    <sigmax>, <sigmay>, <sigmaz> for a single qubit system.

    Parameters: - psi : nparray, wave function at time t (shape: 2,)

    Returns: - r_x_step, float expectation value of x component <sigmax>
             - r_y_step, float expectation value of y component <sigmay>
             - r_z_step, float expectation value of z component <sigmaz>
    """

    # Extract the probability amplitudes for |0> and |1>
    c_0 = psi[0] 
    c_1 = psi[1]
    
    # Complex conjugate of c_0
    c_0_conj = np.conj(c_0) 

    # Bloch vector components derived analytically
    # <sigma_x> = c_0^* c_1 + c_1^* c_0 = 2 * Re(c_0^* * c_1)
    r_x_step = 2 * np.real(c_0_conj * c_1)
    
    # <sigma_y> = -i(c_0^* c_1 - c_1^* c_0) = 2 * Im(c_0^* * c_1)
    r_y_step = 2 * np.imag(c_0_conj * c_1)

    # <sigma_z> = |c_0|^2 - |c_1|^2
    r_z_step = np.abs(c_0)**2 - np.abs(c_1)**2 

    return r_x_step, r_y_step, r_z_step

In [ ]:
def generate_kraus_operators(c_CM, dt, mode="QJ"):
    """
    Generate the Kraus operators M0 and M1 for the single qubit collision model.
    
    Parameters: - c_CM: float, Interaction coefficient
                - dt: float, Time step
                - mode: str, "QJ" for Quantum Jump, "SD" for State Diffusion
                
    Returns: - M0, M1: np.ndarray, The two Kraus operators
    """
    c_dt = c_CM * dt
    cos_val = np.cos(c_dt)
    sin_val = np.sin(c_dt)
    
    if mode == "QJ":
        # Quantum Jump: Measurement in sigma_z basis
        M0 = np.array([[1.0, 0.0], 
                       [0.0, cos_val]], dtype=np.complex128)
                       
        M1 = np.array([[0.0, -1j * sin_val], 
                       [0.0, 0.0]], dtype=np.complex128)
                       
    elif mode == "SD":
        # State Diffusion: Measurement in sigma_x basis (superposition)
        M0 = (1 / np.sqrt(2)) * np.array([[1.0, -1j * sin_val], 
                                          [0.0, cos_val]], dtype=np.complex128)
                                          
        M1 = (1 / np.sqrt(2)) * np.array([[1.0, 1j * sin_val], 
                                          [0.0, cos_val]], dtype=np.complex128)
    else:
        raise ValueError("Mode must be 'QJ' or 'SD'")
        
    return M0, M1

In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def compute_trajectory_wf_core_single(psi_initial, U_site, M0, M1, 
                                      projectors, projectors_cohe,
                                      N_traj, n_times, seeds):
    """
    Core trajectory evolution optimized with Numba for a single qubit.
    Probabilities are dynamically computed at each time step.
    """
    N_state = projectors.shape[0]
    
    # Pre-allocate arrays for N_traj
    pop_traj = np.zeros((N_state, n_times, N_traj), dtype=np.float64)
    coh_traj = np.zeros((N_state, n_times, N_traj), dtype=np.complex128)
    
    # Initialization at t=0
    for i in range(N_state):
        pop_traj[i, 0, :] = np.real(np.vdot(psi_initial, projectors[i] @ psi_initial))
        coh_traj[i, 0, :] = np.vdot(psi_initial, projectors_cohe[i] @ psi_initial)

    for traj in prange(N_traj):
        np.random.seed(seeds[traj])
        psi = psi_initial.copy()

        for step in range(1, n_times):
            # 1. Deterministic evolution given by the isolated System Hamiltonian
            psi = U_site @ psi

            # 2. Apply Kraus operator M1 to test the probability
            v1 = M1 @ psi
            
            # The probability P1 is exactly the squared norm of the resulting vector
            P1 = np.real(np.vdot(v1, v1))
            
            # 3. Stochastic jump
            r = np.random.rand()
            if r < P1:
                psi = v1 # M1 was already applied to v1
            else:
                psi = M0 @ psi

            # 4. State Normalization
            psi = psi / np.linalg.norm(psi)

            # 5. Extract observable values
            for i in range(N_state):
                pop_traj[i, step, traj] = np.real(np.vdot(psi, projectors[i] @ psi))
                coh_traj[i, step, traj] = np.vdot(psi, projectors_cohe[i] @ psi)

    return pop_traj, coh_traj

def compute_trajectory_wf(c_CM, dt, N_traj, times, 
                          projectors, projectors_cohe, psi_sys_initial, U_site, 
                          mode="QJ", batch_size=1000):
    """
    Wrapper function to handle batching and random seeds before calling the JIT core.
    """
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    if psi_sys_initial.ndim > 1:
        psi_sys_initial = psi_sys_initial.flatten()
        
    projectors = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors])
    projectors_cohe = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors_cohe])

    n_times = len(times)
    
    # Generate the specific Kraus Operators
    M0, M1 = generate_kraus_operators(c_CM, dt, mode)

    rng_seeds = np.random.RandomState(42)
    all_seeds = rng_seeds.randint(0, 2**30, size=N_traj)

    pop_00 = np.zeros((n_times, N_traj), dtype=np.float64)
    pop_11 = np.zeros((n_times, N_traj), dtype=np.float64)
    coh_01 = np.zeros((n_times, N_traj), dtype=np.complex128)
    coh_10 = np.zeros((n_times, N_traj), dtype=np.complex128)

    N_done = 0
    n_batches = int(np.ceil(N_traj / batch_size))

    for b in range(n_batches):
        N_batch = min(batch_size, N_traj - N_done)
        seeds_b = all_seeds[N_done : N_done + N_batch]

        pop_b, coh_b = compute_trajectory_wf_core_single(
            psi_sys_initial, U_site, M0, M1, projectors, projectors_cohe,
            N_batch, n_times, seeds_b)

        pop_00[:, N_done : N_done + N_batch] = pop_b[0, :, :]
        pop_11[:, N_done : N_done + N_batch] = pop_b[1, :, :]
        
        coh_01[:, N_done : N_done + N_batch] = coh_b[0, :, :]
        coh_10[:, N_done : N_done + N_batch] = coh_b[1, :, :]

        N_done += N_batch
        del pop_b, coh_b

    return pop_00, pop_11, coh_01, coh_10

In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def compute_trajectory_wf_core_single_low_pop(psi_initial, U_site, M0, M1, 
                                      projectors, projectors_cohe,
                                      N_traj, n_times, seeds):
    """
    Core trajectory evolution optimized with Numba.
    Here, the random number 'r' is scaled by the current state population,
    allowing stable simulation of perturbative regimes.
    """
    N_state = projectors.shape[0]
    
    # Pre-allocate arrays
    pop_traj = np.zeros((N_state, n_times, N_traj), dtype=np.float64)
    coh_traj = np.zeros((N_state, n_times, N_traj), dtype=np.complex128)
    
    # Calculate the initial perturbative norm (e.g., sqrt(10^-3))
    initial_norm = np.linalg.norm(psi_initial)
    
    # Initialization at t=0
    for i in range(N_state):
        pop_traj[i, 0, :] = np.real(np.vdot(psi_initial, projectors[i] @ psi_initial))
        coh_traj[i, 0, :] = np.vdot(psi_initial, projectors_cohe[i] @ psi_initial)

    for traj in prange(N_traj):
        np.random.seed(seeds[traj])
        psi = psi_initial.copy()

        for step in range(1, n_times):
            # 1. Deterministic evolution
            psi = U_site @ psi

            # 2. Apply Kraus operator M1 to get the jump probability
            v1 = M1 @ psi
            
            # P1 is the probability of the jump.
            # Since psi is small, P1 is intrinsically scaled down.
            P1 = np.real(np.vdot(v1, v1))
            
            # 3. Stochastic jump
            # We extract r in the range [0, current_population] instead of [0, 1]
            current_norm_sq = np.real(np.vdot(psi, psi))
            r = np.random.rand() * current_norm_sq
            
            # Now the comparison is mathematically consistent again
            if r < P1:
                # Jump occurred!
                psi = v1 
            else:
                # No jump occurred, apply M0 (deterministic decay)
                psi = M0 @ psi

            # 4. State Renormalization
            # We restore the state to its exact initial perturbative norm, 
            # keeping the dynamics caged in the 10^-3 scale.
            psi = psi * (initial_norm / np.linalg.norm(psi))

            # 5. Extract observable values
            for i in range(N_state):
                pop_traj[i, step, traj] = np.real(np.vdot(psi, projectors[i] @ psi))
                coh_traj[i, step, traj] = np.vdot(psi, projectors_cohe[i] @ psi)

    return pop_traj, coh_traj

    return pop_traj, coh_traj
def compute_trajectory_wf_low_pop(c_CM, dt, N_traj, times, 
                          projectors, projectors_cohe, psi_sys_initial, U_site, 
                          mode="QJ", batch_size=1000):
    """
    Wrapper function to handle batching and random seeds before calling the JIT core.
    """
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    if psi_sys_initial.ndim > 1:
        psi_sys_initial = psi_sys_initial.flatten()
        
    projectors = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors])
    projectors_cohe = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors_cohe])

    n_times = len(times)
    
    # Generate the specific Kraus Operators
    M0, M1 = generate_kraus_operators(c_CM, dt, mode)

    rng_seeds = np.random.RandomState(42)
    all_seeds = rng_seeds.randint(0, 2**30, size=N_traj)

    pop_00 = np.zeros((n_times, N_traj), dtype=np.float64)
    pop_11 = np.zeros((n_times, N_traj), dtype=np.float64)
    coh_01 = np.zeros((n_times, N_traj), dtype=np.complex128)
    coh_10 = np.zeros((n_times, N_traj), dtype=np.complex128)

    N_done = 0
    n_batches = int(np.ceil(N_traj / batch_size))

    for b in range(n_batches):
        N_batch = min(batch_size, N_traj - N_done)
        seeds_b = all_seeds[N_done : N_done + N_batch]

        pop_b, coh_b = compute_trajectory_wf_core_single_low_pop(
            psi_sys_initial, U_site, M0, M1, projectors, projectors_cohe,
            N_batch, n_times, seeds_b)

        pop_00[:, N_done : N_done + N_batch] = pop_b[0, :, :]
        pop_11[:, N_done : N_done + N_batch] = pop_b[1, :, :]
        
        coh_01[:, N_done : N_done + N_batch] = coh_b[0, :, :]
        coh_10[:, N_done : N_done + N_batch] = coh_b[1, :, :]

        N_done += N_batch
        del pop_b, coh_b

    return pop_00, pop_11, coh_01, coh_10


## Main Loop for varying $ dt $ and $ N_{traj} $

In [ ]:
# ===================
# System's Parameters
# ===================
np.random.seed(1) # always use the same seed 
N_site = 3  # Number of sites
#V_array = [1.0]    NO potential
E0 = 0.0  # Energy of the ground state |0>
E1 = 1.5  # Energy of the first excited state |1>
E2 = 2.0  # Energy of the second excited state |2>    

H_Sys = np.diag([E0, E1, E2])  # System Hamiltonian

# =========================
# Time Evolution Parameters
# =========================
dt_list = [0.01]     # change : time step
tf = 100.0    # Final Time
steps_list = [ int(tf / dt_list[i]) for i in range (len(dt_list)) ]
times_list = [ np.linspace(0, tf, int(steps_list[i])) for i in range(len(dt_list))]

N_traj = 20000  # change numberof trajectories

# ===================
# Dephasing Parameter 
# ===================
gamma_r = 0.1   # Gamma rate for the decay
# Lindblad Rates list
gamma_k = [gamma_r ]

# Scaling for the collsional algorithm c = sqrt(gamma / dt)
c_CM_list = np.array([np.sqrt(gamma_r / dt_list[j]) for j in range(len(dt_list))] )  


In [ ]:
# ========================================
# Initial wave function and density matrix
# ========================================

# ======
# System
# ======
pop_0 = np.sqrt(1 - 10**(-3)) # Population in |0> is close to 1, but not exactly 1 to avoid numerical issues
pop_1 = 0.0
pop_2 = np.sqrt(10**(-3))

psi_sys_initial = np.array([pop_0, pop_1, pop_2], dtype=complex) # System is initialized in |2>
rho_sys_initial = np.outer(psi_sys_initial, psi_sys_initial.conj()) # Density matrix of the system at t=0     

# =======
# Ancilla
# =======
# Ancilla is strictly initialized in |0> 
psi_anc_single = np.array([1.0, 0.0], dtype=complex)  # ancilla initialized in |0> always
rho_anc_single = np.outer(psi_anc_single, psi_anc_single.conj())

# =========
# Projectors
# =========
P00 = np.array([[1, 0, 0],
                 [0, 0, 0], 
                 [0, 0, 0]], dtype=complex) # Projector on |0><0|

P11 = np.array([[0, 0, 0], 
                [0, 1, 0], 
                [0, 0, 0]], dtype=complex) # Projector on |1><1|

P22 = np.array([[0, 0, 0], 
                [0, 0, 0], 
                [0, 0, 1]], dtype=complex) # Projector on |2><2|

P12 = np.array([[0, 0, 0], 
                [0, 0, 1], 
                [0, 0, 0]], dtype=complex) # Projector on |1><2|

P21 = np.array([[0, 0, 0], 
                [0, 0, 0], 
                [0, 1, 0]], dtype=complex) # Projector on |2><1|


projectors = np.array([P00, P11, P22], dtype=complex) 
projectors_cohe = np.array([P12, P21], dtype=complex) 

# ======================
# Lindblad Jump Operator
# ======================
L_r = P12 # Jump operator for relaxation |1><2| 
L_k = [L_r]

# =================
# Simulation Modes
# =================
mode_list = ["QJ", "SD"] # Quantum Jump and State Diffusion

### Calculation

In [ ]:
# ======================
# Output directory setup
# ======================
results_dir = "../../Results/Data/Complete_rho/"
os.makedirs(results_dir, exist_ok=True)

# Trajectories per batch in the wrapper
BATCH_SIZE = 1000

def _make_fname_npz(results_dir, mode, dt, N_traj):
    dt_str = f"{dt:.6f}".replace(".", "p")
    return os.path.join(results_dir, f"result_mode_{mode}_dt{dt_str}_Ntraj{N_traj}.npz")

def _already_done_npz(results_dir, mode, dt, N_traj):
    return os.path.isfile(_make_fname_npz(results_dir, mode, dt, N_traj))

# =====================
# Main computation loop
# =====================
print("Starting computation for different modes and dt")

for mode_idx, mode in enumerate(mode_list):

    print("=" * 50)
    print(f"MODE = {mode} ({mode_idx+1}/{len(mode_list)})")
    print("=" * 50)

    for dt_idx, dt in enumerate(dt_list):

        print("=" * 40)
        print(f"dt = {dt:.4f} ({dt_idx+1}/{len(dt_list)})")
        
        # # Check if already computed before running the baselines
        # if _already_done_npz(results_dir, mode, dt, N_traj):
        #     print(f"  Already done for N_traj={N_traj}, skipping.")
        #     continue

        times = times_list[dt_idx]
        steps = steps_list[dt_idx]
        c_CM  = c_CM_list[dt_idx]

        print("Recalculating Hamiltonians")
        H_site, H_coll, H_tot = complete_Hamiltonian(Delta_E, c_CM)
        U_tot, U_diag, w, V = evolution_operator(H_tot, dt, method='diagonalization', hermitian=True)
        U_site, U_diag_site, w_site, V_site = evolution_operator(H_site, dt, method='diagonalization', hermitian=True)

        print("Computing Lindblad")
        start_time = time.time()
        rho_list_lindblad, V_lindblad, W_lindblad = Lindblad_evo(
            rho_sys_initial, H_site, gamma_k, L_k, times, method="diagonal", vectorized=False
        )
        print(f"Completed in {time.time() - start_time:.2f}s")

        print("Computing Trajectory Isolated")
        start_time = time.time()
        pop_traj_isolated, coh_traj_isolated = compute_trajectory_wf_isolated(
            times, projectors, projectors_cohe, psi_sys_initial, U_site
        )
        print(f"Completed in {time.time() - start_time:.2f}s")

        print("Computing Trace Ancilla")
        start_time = time.time()
        pops_trace, cohe_trace = compute_trace_ancilla(
            rho_sys_initial, rho_anc_single, U_diag, V, times, projectors, projectors_cohe
        )
        print(f"Completed in {time.time() - start_time:.2f}s")

        print("-" * 40)
        print(f"Computing Trajectory WF ({mode}) for N_traj = {N_traj}")
        start_time = time.time()

        # Population = 1.0 version
        pop_00, pop_11, coh_01, coh_10 = compute_trajectory_wf(
            c_CM, dt, N_traj, times,
            projectors, projectors_cohe, psi_sys_initial, U_site,
            mode=mode, batch_size=BATCH_SIZE
        )

        # Low population version
        # pop_00, pop_11, coh_01, coh_10 = compute_trajectory_wf_low_pop(
        #     c_CM, dt, N_traj, times,
        #     projectors, projectors_cohe, psi_sys_initial, U_site,
        #     mode=mode, batch_size=BATCH_SIZE
        # )

        print(f"Completed in {time.time() - start_time:.2f}s")
        
        # Pop = 1.0 version
        file_name = "pop_1.0_result_mode_QJ_dt0p010000_Ntraj20000.npz" if mode == "QJ" else "pop_1.0_result_mode_SD_dt0p010000_Ntraj20000.npz"
        fname_npz = os.path.join(results_dir, file_name)

        # Low population version
        #fname_npz = _make_fname_npz(results_dir, mode, dt, N_traj)

        np.savez_compressed(
            fname_npz,
            # 1. Raw Trajectory Data
            pop_00=pop_00,
            pop_11=pop_11,
            coh_01=coh_01, 
            coh_10=coh_10,
            
            # 2. Analytical Baselines & Trace
            pops_trace=pops_trace,
            cohe_trace=cohe_trace,
            rho_list_lindblad=rho_list_lindblad,
            V_lindblad=V_lindblad,
            W_lindblad=W_lindblad,
            
            # 3. Isolated System Data
            pop_traj_isolated=pop_traj_isolated,
            coh_traj_isolated=coh_traj_isolated,
            
            # 4. Parameters
            mode=mode, dt=dt, N_traj=N_traj,
            times=times, steps=steps, c_CM=c_CM
        )

        size_mb = os.path.getsize(fname_npz) / (1024**2)
        print(f"  Saved -> {os.path.basename(fname_npz)}  ({size_mb:.2f} MB)")

        # Immediate RAM Cleanup
        del pop_00, pop_11, coh_01, coh_10
        del rho_list_lindblad, pop_traj_isolated, coh_traj_isolated, pops_trace, cohe_trace

print("\n" + "=" * 40)
print("COMPUTATION COMPLETED!")
print(f"Results saved for:")
print(f"  - {len(mode_list)} modes: {mode_list}")
print(f"  - {len(dt_list)} dt values: {dt_list}")
print(f"  - Fixed N_traj: {N_traj}")
print("=" * 40)